# Data Project Notebook

This notebook is the result of the Data proffesional portfolio requirements. 

The aim of this notebook is to explore TV Show data to investigate whether there are characteristic aassociated with highly-rated and engaging cotent. These findings are then used to inform content acquisition strategies and potentially strategic partnerships.  

The datasets used are from both TMDb and IMDb data. These provide a mix of both TV programme data and user engagement metrics.

Sections have been labelled for clear use and understanding.

## Hypothesis

* H1: Does higher audience engagement correlate with higher content quality?
* H2: Do genres influence a shows content success?
* H3: Can programme data be used to predict content success? 

## HOW TO RUN
Simply execute in your python environment or user space. 

## CONTENTS
* [1. Notebook requirements section](#requirements)
    * [1.1. IMDb data genration](#create)
    * [1.2. TMDb data generation](#clean)
    * [1.3. Merging IMDb and TMDb](#merge)
* [2. Merged Table Exploration](#section)
    * [2.1. Merged Table EDA](#eda)
* [3. Feature Engineering](#conclusions)
    * [3.1. Log Transformations](#log)
    * [2.2. Custom metric creation](#custom) 
* [4. Hypothesis 1](#h1)
* [5. Hypothesis 2](#h2)
    * [5.1. Content Success Creation](#success)
    * [5.2. Genre engineering](#genre)
    * [5.3. Genre performance evaluation](#gvis)
* [6. Hypothesis 3](#h3)
    * [6.1. Feeature engineering enhancement](#log2)
    * [6.2. Genre and Production company encoding](#encode)
    * [6.3. Model fitting and evaluation](#subsection)

## 1. Notebook Requirements and Data Generation
<a class="anchor" id="requirements"></a>

This section focuses executing the notebook requirements and generating the datasets that will be used for analysis.

In [ ]:
# secction for all imports used within the notebook

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


from sklearn.preprocessing import StandardScaler

from scipy.stats import pearsonr
from scipy.stats import f_oneway

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error

import statsmodels.api as sm

from collections import Counter


###  1.1 IMDB Dataset creation and join
<a class="anchor" id="create"></a>

In [ ]:
# Load IMDB title information
imdb_basics = pd.read_csv(
    "title.basics.tsv",
    sep="\t",
    na_values="\\N"
)

# Load IMDB ratings
imdb_ratings = pd.read_csv(
    "title.ratings.tsv",
    sep="\t"
)

print(imdb_basics.head())
print(imdb_ratings.head())

In [ ]:
# Join IMDB data together using the tconst value
imdb_joined = imdb_basics.merge(imdb_ratings, on = "tconst", how = "inner")

# Breakdown of each column type
imdb_joined.info()

# Describe the joined dataset to view key summary statistics
imdb_joined.describe()

In [ ]:
imdb_joined.head()

In [ ]:
# Drop null values from the primary title
imdb_test = imdb_joined.dropna(subset = "primaryTitle")

# Investigate the primary title field to see if multiple title with same name exist
imdb_test = imdb_test[
    imdb_test['primaryTitle'] == "Game of Thrones"

]
imdb_test.head(20)

As shown above there are multiple titles with the exact same name. These will be handled appropriately shortly.

In [ ]:
# Print unique values for the type content in the imdb datasets
print(imdb_joined['titleType'].unique())

In [ ]:
# Create a list of TV content to keep
keep_types = [
    "tvSeries",
    "tvMiniSeries",
    "tvShort",
    "tvSpecial"
]

# Filter IMDb dataset to only TV content
imdb_joined = imdb_joined[imdb_joined["titleType"].isin(keep_types)]

In [ ]:
# identify how many duplicate records exist in the primary title column
imdb_joined["primaryTitle"].duplicated().sum()

In [ ]:
# Drop duplicates based on which has the higher number of user votes
imdb_clean = (
    imdb_joined
    .sort_values("numVotes", ascending=False)
    .drop_duplicates(subset="primaryTitle", keep="first")
)

In [ ]:
# Double check number of duplicate records have decreased
imdb_clean["primaryTitle"].duplicated().sum()

### 1.2 TMDb Dataset generation and cleaning
<a class="anchor" id="clean"></a>

In [ ]:
# Load TMDb dataset
tmdb_data = pd.read_csv(
    "TMDB_tv_dataset_v3_CSV.csv"
)

# Get info in the number of null values and the type of data stored
tmdb_data.info()

# Get descriptive statistics on the tmdb dataset
tmdb_data.describe()

In [ ]:
# See example of what tmdb data is
tmdb_data.head()

In [ ]:
# Apply filtering on the tmdb dataset

# Convert the first air date to a datetime column
tmdb_data["first_air_date"] = pd.to_datetime(
    tmdb_data["first_air_date"],
    dayfirst=True,
    errors="coerce"
)

# Retrieve year from the first air date column
tmdb_data["first_year"] = tmdb_data["first_air_date"].dt.year

# Filter results for shows that were first aired from 2015 onwards 
tmdb_data = tmdb_data[
    tmdb_data["first_year"] >= 2015
]

# Filter results to those that equal to or less than 11 seasons
tmdb_data = tmdb_data[
    tmdb_data["number_of_seasons"] <= 11
]

# Filter shows that have at least 5 votes
tmdb_data = tmdb_data[
    tmdb_data["vote_count"] >= 5
]

tmdb_data.head()

In [ ]:
# Drop null values from both genres and the spoken language column
tmdb_data = tmdb_data.dropna(subset=["spoken_languages", "genres"])

# Filter for shows that are spoken in English
tmdb_cleaned = tmdb_data[
    tmdb_data["spoken_languages"].str.contains(
        "English",
        na=False
    )
].copy()

### 1.3 Merging IMDB and TMDB Dataset
<a class="anchor" id="merge"></a>

In [ ]:
# Make sure that the merging column is cleaned and roughly identical across both datasets
imdb_clean["primaryTitle"] = imdb_clean["primaryTitle"].str.strip().str.lower()
tmdb_cleaned["name"] = tmdb_cleaned["name"].str.strip().str.lower()

In [ ]:
# Specify which columns to keep from the IMDB dataset
imdb_subset = imdb_clean[[
    "primaryTitle",
    "isAdult",
    "startYear",
    "averageRating",
    "numVotes"
]]

In [ ]:
# Specify which columns to keep from the TMDB dataset
tmdb_subset = tmdb_cleaned[[
    "id",
    "name",
    "number_of_seasons",
    "number_of_episodes",
    "original_language",
    "vote_count",
    "vote_average",
    "first_year",
    "popularity",
    "type",
    "genres",
    "networks",
    "production_companies"
]]

In [ ]:
# Merge the two datasets based on the show name for each dataset, keeping only matching results from both
merged_df = pd.merge(
    tmdb_subset,
    imdb_subset,
    left_on="name",
    right_on="primaryTitle",
    how="inner"
)

## 2. Merged Table Exploration
<a class="anchor" id="exp"></a>

This section focuses on exploring the merged dataset and evaluating integrity.

In [ ]:
# Code to see some example records from the merged dataset
merged_df.head()

### 2.1 Merged table EDA
<a class="anchor" id="eda"></a>

In [ ]:
# Return info on the merged dataset on the type of columns and non-null values
merged_df.info()

# Return descriptive statistics for the dataset
merged_df.describe()

In [ ]:
# Return which columns are null with their respective null counts
merged_df.isna().sum()[merged_df.isna().sum() > 0]

In [ ]:
# For the TV shows that have null values for production companies show their respective values
merged_df[
    merged_df["production_companies"].isna()
].sort_values("numVotes", ascending=False)

The number of values missing information for production companies are fairly high representing a little less that 20% of results. As the other information is still greatly values for the other hypothesis the data will be retained and only filtered when necessary for the predictive model.

In [ ]:
# keep only relevant rows for future analysis
merged_lean = merged_df[[
    "id",
    "name",
    "number_of_seasons",
    "number_of_episodes",
    "original_language",
    "vote_count",
    "vote_average",
    "first_year",
    "popularity",
    "type",
    "genres",
    "networks",
    "production_companies",
    "isAdult",
    "averageRating",
    "numVotes"
]].copy()

merged_lean.info()

In [ ]:
# Identify the percentages for the missing values across each column
missing_percent = (
    merged_lean.isna().mean()*100
).sort_values(
    ascending=False
)

missing_percent

In [ ]:
# Create a histogram of average rating to see distribution of imdb TV show ratings
sns.histplot(
    merged_lean["averageRating"],
    bins=20
)

plt.title("Distribution of IMDb Ratings")

## 3. Feature Engineering
<a class="anchor" id="feat"></a>

This sections develops some of the features that will be used in future analytical steps.

### 3.1 Applying Log transformations to Vote count
<a class="anchor" id="log"></a>

In [ ]:
# histogram see the distributions of the TV Show User Votes
sns.histplot(
    merged_lean["numVotes"],
    bins=20
)

plt.title("Distribution of IMDb Number of Votes")

Above you can see that there is an extreme postive skew to the data. To account for this variance a log transformation was applied to the number of votes.

In [ ]:
# Apply logarithmic transforrmation to both IMDb and TMDb user vottes to TV shows
merged_lean['log_imdb_votes'] = np.log1p(merged_lean['numVotes'])
merged_lean['log_tmdb_votes'] = np.log1p(merged_lean['vote_count'])

In [ ]:
# Plot the logarithmic vote count columns to see how the distributions were affected by the transformations
sns.histplot(
    merged_lean["log_imdb_votes"],
    bins=20
)

plt.title("Distribution of IMDb Log Votes")

### 3.2 Custom Metric Calculations
<a class="anchor" id="custom"></a>

In [ ]:
# Call a standard scaler function
scaler = StandardScaler()

# Apply the standard scaler to all metric needed to calculate the custom figures
merged_lean["imdb_rating_scaled"] = scaler.fit_transform(
    merged_lean[["averageRating"]]
)

merged_lean["tmdb_rating_scaled"] = scaler.fit_transform(
    merged_lean[["vote_average"]]
)

merged_lean["imdb_votes_scaled"] = scaler.fit_transform(
    merged_lean[["log_imdb_votes"]]
)

merged_lean["tmdb_votes_scaled"] = scaler.fit_transform(
    merged_lean[["log_tmdb_votes"]]
)

In [ ]:
# Create a quick correlation table between the different features
merged_lean[
    ["imdb_rating_scaled", "tmdb_rating_scaled", "imdb_votes_scaled", "tmdb_votes_scaled"]
].corr()

In [ ]:
# Generate a quality score based on IMDb and TMDb rating with a higher weight for IMDb scores
merged_lean["quality_score"] = (
    0.7 * merged_lean["imdb_rating_scaled"]
    +
    0.3 * merged_lean["tmdb_rating_scaled"]
)

In [ ]:
# Generate an engagement score based on IMDb and TMDb user votes with a higher weight for IMDb votes
merged_lean["engagement_score"] = (
    0.7 * merged_lean["imdb_votes_scaled"]
    +
    0.3 * merged_lean["tmdb_votes_scaled"]
)

In [ ]:
# Investigate the skew for each custom metric
print(merged_lean["engagement_score"].skew())
print(merged_lean["quality_score"].skew())

## 4. H1: Correlation between Engagement Score and Quality Score
<a class="anchor" id="h1"></a>
This section evaluates hypothesis 1.

In [ ]:
# Investigate the Pearson relationship between audience engagement and quality score
corr, p_value = pearsonr(
    merged_lean['engagement_score'],
    merged_lean['quality_score']
)

print("Correlation:", corr)
print("P-value:", p_value)


In [ ]:
# Using the linear model to investigate coefficient and the R-squared value
X = merged_lean['engagement_score']

X = sm.add_constant(X)

y = merged_lean['quality_score']

model = sm.OLS(y, X).fit()

print(model.summary())


In [ ]:
# Create a scatterplot the audience engagment and content quality with a line of best fit
plt.figure(figsize=(8,6))

sns.regplot(
    data=merged_lean,
    x='engagement_score',
    y='quality_score',
    scatter_kws={'alpha':0.3},
    line_kws={'color': 'red'}
)

plt.title('Relationship Between Audience Engagement and Content Quality')
plt.xlabel('Engagement score')
plt.ylabel('Quality Score')

plt.show()

## 5. H2: Success scores differentiate between genres
<a class="anchor" id="h2"></a>
This section evalutes hypothesis 2.

In [ ]:
# Showed correlation between audience enagement and quality score
# Generate a success score based on quality and engagement score
merged_lean["success_score"] = (
    0.5 * merged_lean["quality_score"] +
    0.5 * merged_lean["engagement_score"]
)

### 5.1 Success score exploration
<a class="anchor" id="success"></a>

In [ ]:
# Return summary statistics on the success score
merged_lean["success_score"].describe()

In [ ]:
# Create a histogram of the success score

sns.histplot(
    merged_lean["success_score"],
    kde=True
)

plt.title("Distribution of Success Score")

plt.show()

### 5.2 Genre feature engineering step
<a class="anchor" id="genre"></a>

In [ ]:
# Briefly see how genres are distributed, some records appear with multiple genres
merged_lean["genres"]

In [ ]:
# Create a copy of the merged Lean table
genre_df = merged_lean.copy()

# Split the genres by commas
genre_df["genres"] = genre_df["genres"].str.split(",")

genre_df = genre_df.explode("genres")

# Removes trailing spaces form the string genres column
genre_df["genres"] = genre_df["genres"].str.strip()
genre_df

TV Shows now appear multiple times with all the different genres associated with them. Unfortunately as a main genre was not able to be specific this was the next best alternative.

In [ ]:
# Group the results by genre and calculate both count of the times genres appear and the mean success scores
genre_performance = (
    genre_df
    .groupby("genres")
    .agg(
        avg_success=("success_score","mean"),
        show_count=("success_score","count")
    )
    .sort_values("avg_success", ascending=False)
)

In [ ]:
# Filter by genres for those that appear at minimum 25 times
genre_performance_high = genre_performance[
    genre_performance["show_count"] >= 25
]
genre_performance_high

In [ ]:
# Filter by genres for those that appear at less than 25 times
genre_performance_low = genre_performance[
    genre_performance["show_count"] < 25
]
genre_performance_low

In [ ]:
# Removing columns that are in the genres with less than 25 counts
genre_df = genre_df[~genre_df["genres"].isin(["Musical", "Western", "Soap", "Romance", "News"])]


### 5.3 Genre performance visualisations and analysis
<a class="anchor" id="gvis"></a>

In [ ]:
# Create a boxplots of genre performances based on their success scores 

plt.figure(figsize=(14,7))

sns.boxplot(
    data=genre_df,
    x="genres",
    y="success_score"
)

plt.xticks(rotation=45)

plt.title("Success Score Distribution by Genre")

plt.show()

In [ ]:
# Perform a One-way ANOVA of the genre groups to evaluate whether they are statistically different
genre_groups = [
    group["success_score"].values
    for name, group in genre_df.groupby("genres")
]

f_stat, p_value = f_oneway(*genre_groups)

print(f_stat)
print(p_value)

## 6. H3: Success Score Prediction Model
<a class="anchor" id="h3"></a>
This section evaluates hypothesis 3.

### 6.1 Feature engineering Development
<a class="anchor" id="log2"></a>

In [ ]:
# Histogram on the number of seasons
sns.histplot(
    merged_lean["number_of_seasons"],
    bins=20
)

plt.title("Distribution of Number of Seasons")
plt.show()

In [ ]:
# Histogram on the number of episodes
sns.histplot(
    merged_lean["number_of_episodes"],
    bins=20
)

plt.title("Distribution of Number of episodes")
plt.show()

In [ ]:
# Perform a logarithmic transformation on the number of episodes
merged_lean['log_episodes'] = np.log1p(
    merged_lean['number_of_episodes']
)

# Create a histogram of the new log transformed episode column
sns.histplot(
    merged_lean["log_episodes"],
    bins=20
)

plt.title("Distribution of log number of Episodes")
plt.show()

In [ ]:
# Drop null values from production companies
merged_lean.dropna(subset=["production_companies"])

### 6.2 Genre and Production company encoding
<a class="anchor" id="encode"></a>

In [ ]:
# convert column based to a list
merged_lean["genres"] = (
    merged_lean["genres"]
    .apply(
        lambda x: [
            g.strip().title()
            for g in x.split(",")
            if g.strip()
        ]
    )
)

In [ ]:
# Count how often each genre appears and make it into a dataframe
genre_counter = Counter()

for genres in merged_lean["genres"]:
    genre_counter.update(genres)

genre_counts = pd.DataFrame(
    genre_counter.items(),
    columns=["Genre","Count"]
).sort_values(
    "Count",
    ascending=False
)

print(genre_counts)

In [ ]:
# Select genres to keep by retaining ones that appear more than 25 times
selected_genres = [
    genre
    for genre,count
    in genre_counter.items()
    if count >= 25
]

In [ ]:
# Call multi label binarizer, useful when there can be multiple values for one row
# This turns values to 0 and 1 so that it can be used in a model
mlb = MultiLabelBinarizer()

genre_dummies = pd.DataFrame(
    mlb.fit_transform(
        merged_lean["genres"]
    ),
    columns=mlb.classes_,
    index=merged_lean.index
)


In [ ]:
# Filter genres by previously selected values
genre_dummies = genre_dummies[
    selected_genres
]

In [ ]:
genre_dummies.head()

In [ ]:
# convert genre column to a list
merged_lean["production_companies"] = (
    merged_lean["production_companies"]
    .fillna("")
    .apply(
        lambda x: [
            company.strip()
            for company in x.split(",")
            if company.strip()
        ]
    )
)

In [ ]:
# Create a count of each production company
company_counter = Counter()

for companies in merged_lean["production_companies"]:
    company_counter.update(companies)

company_counts = pd.DataFrame(
    company_counter.items(),
    columns=["Company", "Count"]
).sort_values(
    "Count",
    ascending=False
)

company_counts.head(20)

In [ ]:
# Retain companies that at least 15 different shows produced
selected_companies = [
    company
    for company, count
    in company_counter.items()
    if count >= 15
]

In [ ]:
len(selected_companies)

In [ ]:
# Turn the Company values to 0 and 1 to be used in model generation

company_mlb = MultiLabelBinarizer()

company_dummies = pd.DataFrame(
    company_mlb.fit_transform(
        merged_lean["production_companies"]
    ),
    columns=company_mlb.classes_,
    index=merged_lean.index
)

In [ ]:
# Filter companies based on the selected companies list
company_dummies = company_dummies[
    selected_companies
]

### 6.3 Model fitting and prediction
<a class="anchor" id="subsection"></a>

In [ ]:
# Fit the X with the features

X = pd.concat(
    [
        merged_lean[
            [
                "number_of_seasons",
                "log_episodes"
            ]
        ],
        genre_dummies,
        company_dummies
    ],
    axis=1
)

# Fit y with the desired prediction outcome
y = merged_lean["success_score"]

In [ ]:
# Create a train and test split of the model

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Create a linear regression model and fit it to the training split
lr = LinearRegression()

lr.fit(
    X_train,
    y_train
)

In [ ]:
# Predict the x test
y_pred = lr.predict(X_test)

In [ ]:
# Return the r2 value of the model
r2 = r2_score(
    y_test,
    y_pred
)

print(
    "R² =", round(r2,3)
)

# Return the mean absolute error of the model
mae = mean_absolute_error(
    y_test,
    y_pred
)

print(
    "MAE =", round(mae,3)
)

# Return the root mean squared error
rmse = root_mean_squared_error(
    y_test,
    y_pred
)

print(
    "RMSE =", round(rmse,3)
)

In [ ]:
# determine the coefficients of the model
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": lr.coef_
})

In [ ]:
# Return the top 29 coeeficients that had a positive impact on succcess score
coefficients.sort_values(
    "Coefficient",
    ascending=False
).head(20)

In [ ]:
# return the bottom 20 coeffecients that had a negative impact on content success score
coefficients.sort_values(
    "Coefficient"
).head(20)